Let's calculate the middle value of the lamp spectra detecred before image1 spectra (so before 2024-01-14T07:59:59.507).\
They are VIS ThAr lamp spectra of Catherine CCD found in ESO raw data archive.

In [1]:
import glob
import os
from pyraf import iraf

# 0. If you Windows operative system, be sure that in your input file_list.txt there is _ and not :
# also replace the middle point "·" with "_" on the files automatically dowloaded by the ESO archive into C or D Windows hard disks.

# 1. Load the necessary IRAF packages
iraf.noao()
iraf.imred()
iraf.ccdred()

# 2. Make sure to adjust the pattern to match your actual lamp file names
lamp_pattern = "XSHOO*.fits"
lamp_files = sorted(glob.glob(lamp_pattern))

if not lamp_files:
    raise FileNotFoundError(f"No files found with pattern: {lamp_pattern}")

# If the data comes from MEF (multi-extension) instruments like X-Shooter,
# specify the scientific extension for each file (e.g., bias.fits[1])
# bias_files = [f"{f}[1]" for f in bias_files]

input_list = "lamp_list_beforeobs.txt"
#with open(input_list, "w") as f:   #it's writing file names into the list automatically 
#    for filename in lamp_files:
#        f.write(f"{filename}\n")

# 3. Name of the output Master lamp file
output_master_lamp = "lamp_b.fits"

# Remove the previous output if it already exists
if os.path.exists(output_master_lamp):
    os.remove(output_master_lamp)

# 4. Run zerocombine using PyRAF
# Note: combine='median' or 'average' with reject='sigclip' or 'minmax'
iraf.zerocombine(
    input=f"@{input_list}",
    output=output_master_lamp,
    combine="average",
    reject="minmax",
    ccdtype="",   # Leave it empty to avoid filtering on ccdtype header, the default is "zero" but this fits header has different keywords
    process="no",  # Do not process the output (e.g., do not trim or scale)
    scale="none",  # Bias frames typically do not require scaling
    statsec="",  # Use the entire frame if empty
    #interactive="no",
)

print(f"Master lamp created successfully: {output_master_lamp}")

# 5. Cleanup (optional)
#if os.path.exists(input_list):
#    os.remove(input_list)

imred/:
 argus/         ctioslit/       hydra/          kpnocoude/      vtel/
 bias/          dtoi/           iids/           kpnoslit/
 ccdred/        echelle/        irred/          quadred/
 crutil/        generic/        irs/            specred/
ccdred/:
 badpiximage    ccdlist         combine         mkillumcor      setinstrument
 ccdgroups      ccdmask         darkcombine     mkillumflat     zerocombine
 ccdhedit       ccdproc         flatcombine     mkskycor
 ccdinstrument  ccdtest/        mkfringecor     mkskyflat
Master lamp created successfully: lamp_b.fits


My lamps are binned $1\times1$ instead of $2\times1$ like my images and the CCD measurement is not $2106\times2000$ like the image but it's $2106\times4000$.\
So I'm modifiding the binning and the size of my lamp with iraf.

In [2]:
import os
from pyraf import iraf

def rebin_xshooter_vis_lamp(input_fits, output_fits):
    # Load the necessary IRAF packages
    iraf.images(_doprint=0)

    # Remove the output file if it already exists
    if os.path.exists(output_fits):
        os.remove(output_fits)

    print(f"1. Compress 1x2 with blkavg: {input_fits} -> {output_fits}")
    
    # b1=1 (doens't change in X), b2=2 (sum on Y with a step of 2)
    # option='sum' saves the total flux in the new binned pixels, which is important for lamp frames
    iraf.blkavg(
        input=input_fits,
        output=output_fits,
        b1=1,
        b2=2,
        option="sum"
    )

    print("2. Updating the header for ESO compatibility...")

    # Maps of modified and added keywords in the new FITS file
    header_updates = {
        # Binning factors
        "CDELT2": 2.0,
        "HIERARCH ESO DET WIN1 BINY": 2,
        
        # New Y ax dimensions 
        "HIERARCH ESO DET WIN1 NY": 2000,
        "HIERARCH ESO DET OUT1 NY": 2000,
        
        # If present on the original header, update the binning factor for the output image:
        "BINY": 2
    }

    for key, val in header_updates.items():
        try:
            iraf.hedit(
                images=output_fits,
                fields=key,
                value=val,
                add=True,       # Add the key if it was missing, overwrite if it exists
                addonly=False,
                delete=False,
                verify=False,
                show=False,
                update=True
            )
        except Exception as e:
            print(f"Warning on {key}: {e}")

    print(f"Done. New file {output_fits} ready for the pipeline.")

if __name__ == "__main__":
    # Add you files names
    file_source = "lamp_b.fits"
    file_final = "lamp_b_binned.fits"

    rebin_xshooter_vis_lamp(file_source, file_final)

1. Compress 1x2 with blkavg: lamp_b.fits -> lamp_b_binned.fits
2. Updating the header for ESO compatibility...
Done. New file lamp_b_binned.fits ready for the pipeline.


Let's subtract the bias from the lamp:

In [5]:
import os
from pyraf import iraf

# 1. Load packages containing imarith and statistics
iraf.images()
iraf.imutil()

# 2.0 Define relative paths
bias_dir = os.path.join("..", "bias")

# 2.1 Define file names
lamp_raw = "lamp_b_binned.fits"
master_bias = os.path.join(bias_dir, "Bias.fits")

# Intermediate and final files
lamp_sub_bias = "lamp_b_binned_sub_bias.fits"


def remove_if_exists(filepath):
    if os.path.exists(filepath):
        os.remove(filepath)


for f in [lamp_sub_bias]:
    remove_if_exists(f)


# 3. Subtraction of Bias from the lamp (Lamp - Bias)
iraf.imarith(
    operand1=lamp_raw,
    op="-",
    operand2=master_bias,
    result=lamp_sub_bias,
    title="Lamp binned debiased",
)


We can now edit the apertures of the lamp using the image1 reference that is in the parent directory.

In [7]:
from pyraf import iraf

iraf.noao()
iraf.twodspec()
iraf.apextract()

iraf.apextract.database = '../database'

iraf.apsum(
    input='lamp_b_binned_sub_bias.fits',
    output='lamp_b_binned_sub_bias.ms.fits',
    apertures='',
    format='multispec',
    reference='image1_sub_bias',
    profiles='',
    interactive='no',
    find='no',
    recenter='no',
    resize='no',
    edit='no',
    trace='no',
    fittrace='no',
    extract='yes',
    extras='no',
    review='no',
    line='INDEF',
    nsum=10,
    background='none',
    weights='none',
    pfit='fit1d',
    clean='no',
    skybox=1,
    saturation='INDEF',
    readnoise=0.0,
    gain=1.0,
    lsigma=4.0,
    usigma=4.0,
    nsubaps=1,
    mode='ql'
)

We have finally obtained the ThAr VIS lamp spectra with the right binning, without bias and with the right apertures: [lamp_b_binned_sub_bias.ms.fits](lamp_b_binned_sub_bias.ms.fits)